# Module D — Ranking, Scoring, & Evaluation

1. Translation Failures
2. Named Entity Mismatch
3. Semantic vs. Lexical Wins
4. Cross-Script Ambiguity
5. Code-Switching

In [12]:
import feedparser
import json
from tqdm import tqdm
import os
from bs4 import BeautifulSoup
import html
import re
import pickle
from rank_bm25 import BM25Okapi

## 4. BN <-> EN CLIR

### 4.1 Load Saved BM25 Indexes

In [13]:
def load_index(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

en_pack = load_index('bm25_en.pkl')
bn_pack = load_index('bm25_bn.pkl')

bm25_en = en_pack['bm25']
doc_ids_en = en_pack['doc_ids']
docs_en = en_pack['docs']

bm25_bn = bn_pack['bm25']
doc_ids_bn = bn_pack['doc_ids']
docs_bn = bn_pack['docs']


def tokenize_en(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

def tokenize_bn(text):
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

def search_en(query, top_k=5):
    q = tokenize_en(query)
    scores = bm25_en.get_scores(q)
    idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [{"doc_id": doc_ids_en[i], "score": float(scores[i]), "title": docs_en[i].get("title",""), "url": docs_en[i].get("url","")} for i in idx]

def search_bn(query, top_k=5):
    q = tokenize_bn(query)
    scores = bm25_bn.get_scores(q)
    idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [{"doc_id": doc_ids_bn[i], "score": float(scores[i]), "title": docs_bn[i].get("title",""), "url": docs_bn[i].get("url","")} for i in idx]

print("Index loaded:" , len(docs_en), "English documents and", len(docs_bn), "Bengali documents.")


Index loaded: 150 English documents and 146 Bengali documents.


### 4.2 Language Detection (BN vs EN)

In [14]:
def is_bangla(text):
    for ch in text:
        o = ord(ch) #unicode of bangla
        if 0x0980 <= o <= 0x09FF:
            return True
    return False    

In [15]:
is_bangla("এই একটি বাংলা বাক্য।")
is_bangla("this is an english sentence.")

False

### 4.3 Load BN <-> EN translation model (MarianMT / OPUS MT)

In [16]:
from transformers import MarianMTModel, MarianTokenizer

BN_EN_NAME = "Helsinki-NLP/opus-mt-bn-en"
EN_BN_NAME = "shhossain/opus-mt-en-to-bn"

tok_bn_en = MarianTokenizer.from_pretrained(BN_EN_NAME)
mod_bn_en = MarianMTModel.from_pretrained(BN_EN_NAME)

tok_en_bn = MarianTokenizer.from_pretrained(EN_BN_NAME)
mod_en_bn = MarianMTModel.from_pretrained(EN_BN_NAME)

In [17]:
def translate_bn_to_en(text):
    batch = tok_bn_en([text], return_tensors="pt",padding=True, truncation=True)
    gen = mod_bn_en.generate(**batch, max_new_tokens=128)
    return tok_bn_en.batch_decode(gen, skip_special_tokens=True)[0]

print("bn_to_en Translation models loaded.")


bn_to_en Translation models loaded.


In [18]:
def translate_en_to_bn(text):
    batch = tok_en_bn([text], return_tensors="pt",padding=True, truncation=True)
    gen = mod_en_bn.generate(**batch, max_new_tokens=128)
    return tok_en_bn.batch_decode(gen, skip_special_tokens=True)[0]

print("en_to_bn Translation models loaded.")

en_to_bn Translation models loaded.


In [19]:
print(translate_bn_to_en("বাংলাদেশ একটি সুন্দর দেশ।"))
print(translate_en_to_bn("Bangladesh is a beautiful country."))

Bangladesh is a beautiful country.
বাংলাদেশ একটি সুন্দর দেশ।


### 4.4 CLIR Search Function

In [20]:
def clir_search(query, top_k=5):
    if is_bangla(query):
        q_en = translate_bn_to_en(query)
        results_bn = search_bn(query, top_k)
        results_en = search_en(q_en, top_k)
        return_en = {"queary_language": "bn", "translated_query": q_en, "results_language": "en", "results": results_en}
        return_bn = {"queary_language": "bn", "translated_query": q_en, "results_language": "bn", "results": results_bn}
        return return_bn,return_en
    else:
        q_bn = translate_en_to_bn(query)
        results_en = search_en(query, top_k)
        results_bn = search_bn(q_bn, top_k)
        return_bn = {"queary_language": "en", "translated_query": q_bn, "results_language": "bn", "results": results_bn}
        return_en = {"queary_language": "en", "translated_query": q_bn, "results_language": "en", "results": results_en}
        return return_bn,return_en

### 4.5 Test

In [21]:
query_key= "বাংলাদেশ ক্রিকেট"
out_bn, out_en = clir_search(query_key, top_k=5)

print("Print result in Bangla:")
for r in out_bn['results']:
    print(r['score'],r['title'],"-", r['url'])
    print("____________________________________________")


print("Result in English:")
for r in out_en['results']:
    print(r['score'],r['title'],"-", r['url'])
    print("____________________________________________")

Print result in Bangla:
7.399980740492559 ৪ বছর পর বিপিএলে শান্তর সেঞ্চুরি - https://www.risingbd.com/sports/news/633526
____________________________________________
4.819759886943764 চট্টগ্রামের কাছে পাত্তা পেল না নবাগত নোয়াখালী এক্সপ্রেস - https://www.risingbd.com/sports/news/633563
____________________________________________
4.793073593425571 বেগের ব্যাটে চ্যালেঞ্জিং সংগ্রহ পেল চট্টগ্রাম - https://www.risingbd.com/sports/news/633554
____________________________________________
4.149732889662082 টস জিতে শান্তর রাজশাহীকে ব্যাটিংয়ে পাঠালো মিঠুনের ঢাকা - https://www.jagonews24.com/sports/cricket/1079254
____________________________________________
3.3866996719499656 ৭ রানে ৮ উইকেট, আন্তর্জাতিক টি-টোয়েন্টিতে বিশ্বরেকর্ড - https://www.jagonews24.com/sports/cricket/1079231
____________________________________________
Result in English:
6.416134376204756 the year 2025 will be remembered less for what bangladesh cricket achieved on the field and more for the turbulence that engulfed it off 

In [22]:
query_key= "Bangladesh Cricket"
out_bn, out_en = clir_search(query_key, top_k=5)

print("Print result in Bangla:")
for r in out_bn['results']:
    print(r['score'],r['title'],"-", r['url'])
    print("____________________________________________")


print("Result in English:")
for r in out_en['results']:
    print(r['score'],r['title'],"-", r['url'])
    print("____________________________________________")

Print result in Bangla:
7.399980740492559 ৪ বছর পর বিপিএলে শান্তর সেঞ্চুরি - https://www.risingbd.com/sports/news/633526
____________________________________________
4.819759886943764 চট্টগ্রামের কাছে পাত্তা পেল না নবাগত নোয়াখালী এক্সপ্রেস - https://www.risingbd.com/sports/news/633563
____________________________________________
4.793073593425571 বেগের ব্যাটে চ্যালেঞ্জিং সংগ্রহ পেল চট্টগ্রাম - https://www.risingbd.com/sports/news/633554
____________________________________________
4.149732889662082 টস জিতে শান্তর রাজশাহীকে ব্যাটিংয়ে পাঠালো মিঠুনের ঢাকা - https://www.jagonews24.com/sports/cricket/1079254
____________________________________________
3.3866996719499656 ৭ রানে ৮ উইকেট, আন্তর্জাতিক টি-টোয়েন্টিতে বিশ্বরেকর্ড - https://www.jagonews24.com/sports/cricket/1079231
____________________________________________
Result in English:
6.416134376204756 the year 2025 will be remembered less for what bangladesh cricket achieved on the field and more for the turbulence that engulfed it off 